# vLLM vs SGLang on a Colab GPU
Runtime → Change runtime type → **L4 GPU**, then Run all.

- Installs each engine in its own virtualenv (about 10 minutes).
- Runs a quick smoke test (about 10 minutes), then the full sweep (about 50 minutes).
- Total is roughly 1.5 hours, about 3 compute units. Background execution is fine.
- At the end it downloads `serving_results.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.isdir('/content/lowbit-decode-gpu'):
    !git clone -q https://github.com/FredWen-S/lowbit-decode-gpu.git /content/lowbit-decode-gpu
%cd /content/lowbit-decode-gpu
!git pull -q && git log -1 --format='%h %s'

Each engine gets its own virtualenv, because their torch pins conflict. The load generator runs in Colab's own Python.

In [ ]:
!pip -q install uv
!uv venv -q --python 3.12 /content/venv-vllm && VIRTUAL_ENV=/content/venv-vllm uv pip install -q vllm
!uv venv -q --python 3.12 /content/venv-sglang && VIRTUAL_ENV=/content/venv-sglang uv pip install -q 'sglang[all]'
!/content/venv-vllm/bin/python -c 'import vllm; print("vllm", vllm.__version__)'
!/content/venv-sglang/bin/python -c 'import sglang; print("sglang", sglang.__version__)'

In [ ]:
from huggingface_hub import snapshot_download
MODEL = snapshot_download('Qwen/Qwen2.5-1.5B-Instruct', local_dir='/content/models/qwen2.5-1.5b-instruct')
!pip -q install aiohttp
print(MODEL)

Smoke test: every config on both engines, with few requests. If a server fails to start, its log is in `results/serving_quick/<engine>/<config>/server.log`.

In [ ]:
!python bench/serve_sweep.py --model /content/models/qwen2.5-1.5b-instruct --python vllm=/content/venv-vllm/bin/python --python sglang=/content/venv-sglang/bin/python --quick --out results/serving_quick

Full sweep.

In [ ]:
!python bench/serve_sweep.py --model /content/models/qwen2.5-1.5b-instruct --python vllm=/content/venv-vllm/bin/python --python sglang=/content/venv-sglang/bin/python --out results/serving

In [ ]:
!cd results && zip -qr /content/serving_results.zip serving serving_quick
from google.colab import files
files.download('/content/serving_results.zip')